In [124]:
!pip install moviepy pandas seaborn matplotlib


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: C:\Users\dell\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [125]:
##Checking if video is loaded properly
from moviepy.video.io.VideoFileClip import VideoFileClip #imports VideoFileClip class from the moviepy.video.io module
video=VideoFileClip(r"D:\content_generation_system\data\ai_podcast.mp4")
print("Video Duration in seconds: ", video.duration)

Video Duration in seconds:  1912.81


In [126]:

##Extraction of Audio
audio_path=r"D:\content_generation_system\audio\ai_podcast_audio.mp3"
video.audio.write_audiofile(audio_path, bitrate="64k")
print("Audio extracted successfully")

chunk:  29%|██▉       | 103/353 [13:58<00:00, 281.06it/s, now=None]

MoviePy - Writing audio in D:\content_generation_system\audio\ai_podcast_audio.mp3


chunk:  29%|██▉       | 103/353 [14:45<00:00, 281.06it/s, now=None]

MoviePy - Done.
Audio extracted successfully


In [127]:
!pip install google-genai


[notice] A new release of pip is available: 23.0.1 -> 26.0.1
[notice] To update, run: C:\Users\dell\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.10_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [128]:
##We are using the AI Model Gemini 1.5 Flash provided by Google. API key is a unique key that 
##lets the code to access online services. This model requires sppech understanding, topic segmentation, title 
##generation, so instead of building complex ML Model, we let Gemini perform the analysis.
from google import genai
import json
import os
from dotenv import load_dotenv
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
client = genai.Client(api_key=api_key)
print("Client initialized safely!")


Client initialized safely!


In [129]:
##Uploading the Audio
import time

audio_file = client.files.upload(file=audio_path)
print(f"Audio uploaded as: {audio_file.name}")

while audio_file.state.name == "PROCESSING":
    print("Waiting for audio to process...")
    time.sleep(2)
    audio_file = client.files.get(name=audio_file.name)

print("Audio ready!")

Audio uploaded as: files/lnxih9dz1re2
Audio ready!


In [135]:
prompt =  """
Analyze this podcast and identify the top 5 short-form clip candidates
suitable for Reels, Shorts, TikTok.

Rules:
- Duration: 20-45 seconds
- Only include meaningful insights, practical advice, or strong opinions
- Skip intros, greetings, filler, or sponsor messages

Return JSON array with:
start, end, topic, content_type, title, description, confidence
"""

In [136]:
import json
from google.genai import errors

MODEL_ID = "gemini-2.5-flash-lite"

def generate_clips_with_retry(audio_file, prompt, max_retries=5):
    for i in range(max_retries):
        try:
            print(f"\n Attempt {i+1}: Sending request...")

            response = client.models.generate_content(
                model=MODEL_ID,
                contents=[audio_file, prompt],
                config={"response_mime_type": "application/json"}
            )

            print(" Success!")

            print("\nRAW RESPONSE:")
            print(response.text)

            parsed = json.loads(response.text)
            return parsed

        except Exception as e:
            print(f"\nError: {e}")

            if "503" in str(e) or "429" in str(e):
                time.sleep(2 ** i)
            else:
                break

    return None
metadata = generate_clips_with_retry(audio_file, prompt)
print("\n====== RESULT ======\n")

if metadata:
    for clip in metadata:
        print(clip)
else:
    print(" No data returned")



 Attempt 1: Sending request...
 Success!

RAW RESPONSE:
[
  {
    "start": "00:11",
    "end": "00:33",
    "topic": "Office Hours",
    "content_type": "Advice",
    "title": "Professor's Office Hours Explained",
    "description": "The host explains the concept of office hours in a university setting, where students can meet with professors to review material and ask questions in more detail.",
    "confidence": 0.8
  },
  {
    "start": "01:38",
    "end": "02:04",
    "topic": "Morning Routine",
    "content_type": "Advice",
    "title": "Essential Morning Protocols",
    "description": "The host outlines his morning routine, starting with waking up, writing down the time, and the importance of sunlight exposure. He emphasizes optimizing sleep, learning, and creativity.",
    "confidence": 0.9
  },
  {
    "start": "02:05",
    "end": "02:31",
    "topic": "Morning Walk",
    "content_type": "Advice",
    "title": "The Power of a Morning Walk",
    "description": "The host explain

In [137]:
#Cleaning and Structuring Data
import uuid

# Helper to convert MM:SS to seconds
def timestamp_to_seconds(ts):
    parts = [int(p) for p in ts.split(":")]
    return parts[0]*60 + parts[1] if len(parts) == 2 else parts[0]*3600 + parts[1]*60 + parts[2]

processed_clips = []

for clip in metadata:
    start = timestamp_to_seconds(clip["start"])
    end = timestamp_to_seconds(clip["end"])
    duration = end - start

    processed_clips.append({
        "clip_id": str(uuid.uuid4()),
        "start_time": float(start),
        "end_time": float(end),
        "duration": float(duration),
        "topic": clip.get("topic", "General"),
        "content_type": clip.get("content_type", "Discussion"),
        "title": clip.get("title", "Untitled Clip"),
        "description": clip.get("description", "No description provided"),
        "confidence_score": float(clip.get("confidence", 0.0))
    })

# Extract only the top candidates for your video generation block later
top_clips = [c for i, c in enumerate(processed_clips) if metadata[i].get("is_top_candidate") == True]

if not top_clips:
    top_clips = sorted(processed_clips, key=lambda x: x['confidence_score'], reverse=True)[:5]

print(f"Successfully processed {len(processed_clips)} segments.")
print(f"Top {len(top_clips)} clips selected for rendering.")

Successfully processed 5 segments.
Top 5 clips selected for rendering.


In [138]:
import json

json_path = r"D:\content_generation_system\clips_metadata.json"

with open(json_path, "w") as f:
    json.dump(processed_clips, f, indent=4)

print("JSON saved at:", json_path)

JSON saved at: D:\content_generation_system\clips_metadata.json


In [139]:
import pandas as pd

df = pd.DataFrame(processed_clips)

csv_path = r"D:\content_generation_system\clips_metadata.csv"
df.to_csv(csv_path, index=False)

print("CSV saved at:", csv_path)

CSV saved at: D:\content_generation_system\clips_metadata.csv


In [140]:
#Generate Video Clips
import os
import re
from moviepy.video.io.VideoFileClip import VideoFileClip


video_path = r"D:\content_generation_system\data\ai_podcast.mp4"
output_folder = r"D:\content_generation_system\clips"
os.makedirs(output_folder, exist_ok=True)

print("--- Starting Video Export ---")

with VideoFileClip(video_path) as video:
    for i, clip_data in enumerate(processed_clips):
        start = clip_data["start_time"]
        end = clip_data["end_time"]

        
        clean_title = re.sub(r'[\\/*?:"<>|]', "", clip_data['title'])
        clean_title = clean_title.replace(' ', '_')

        
        filename = f"{i+1}_{clean_title[:30]}.mp4"
        path = os.path.join(output_folder, filename)

        print(f"Exporting Clip {i+1}: {filename}")

        
        subclip = video.subclipped(start, end)
        
        subclip.write_videofile(
            path, 
            codec="libx264", 
            audio_codec="aac", 
            bitrate="5000k", 
            preset="medium",
            logger=None    
        )

print("\n--- ALL CLIPS SAVED SUCCESSFULLY ---")

--- Starting Video Export ---
Exporting Clip 1: 1_Professor's_Office_Hours_Expla.mp4
Exporting Clip 2: 2_Essential_Morning_Protocols.mp4
Exporting Clip 3: 3_The_Power_of_a_Morning_Walk.mp4
Exporting Clip 4: 4_Why_Morning_Sunlight_is_Vital.mp4
Exporting Clip 5: 5_Boosting_Alertness_and_Calm.mp4

--- ALL CLIPS SAVED SUCCESSFULLY ---
